In [ ]:
import googlemaps
import re
from datetime import datetime

# 1. Inicializar el cliente con tu API Key
# Reemplaza 'TU_API_KEY_AQUI' con la clave real que obtuviste en Google Cloud
API_KEY = 'TU_API_KEY_AQUI'
gmaps = googlemaps.Client(key=API_KEY)

def calcular_ruta(origen, destino, modo_transporte="walking"):
    print(f"Calculando ruta desde '{origen}' hasta '{destino}'...\n")
    
    # 2. Solicitar la ruta a la API (Directions API)
    now = datetime.now()
    try:
        resultado = gmaps.directions(origen,
                                     destino,
                                     mode=modo_transporte, # Puede ser "driving", "walking", "bicycling" o "transit"
                                     departure_time=now,
                                     language="es") # Resultados en español
        
        # 3. Procesar la respuesta
        if resultado:
            # Extraemos la información del primer trayecto (leg)
            ruta = resultado[0]['legs'][0]
            distancia = ruta['distance']['text']
            duracion = ruta['duration']['text']

            print("-" * 40)
            print(f"Distancia total: {distancia}")
            print(f"Tiempo estimado: {duracion}")
            print("-" * 40)
            print("Instrucciones paso a paso:")

            # Iteramos sobre los pasos para mostrarlos
            for paso in ruta['steps']:
                # La API devuelve el texto con etiquetas HTML (ej. <b>Gira a la derecha</b>)
                # Usamos una expresión regular simple para limpiar ese HTML y dejar solo el texto
                instruccion_limpia = re.sub(r'<[^>]+>', '', paso['html_instructions'])
                distancia_paso = paso['distance']['text']
                
                print(f" > {instruccion_limpia} ({distancia_paso})")
        else:
            print("No se encontró ninguna ruta entre esos dos puntos.")
            
    except Exception as e:
        print(f"Ocurrió un error al conectar con la API: {e}")

# --- Ejecución del programa ---
if __name__ == "__main__":
    # Puedes usar direcciones exactas, nombres de ciudades, o lugares emblemáticos
    mi_ubicacion = "Valencia, España" 
    mi_destino = "Ciudad de las Artes y las Ciencias, Valencia"
    
    calcular_ruta(mi_ubicacion, mi_destino)

In [3]:
import time
import os
from gtts import gTTS
import pygame
from geopy.distance import geodesic

# 1. Inicializar el mezclador de audio de pygame
pygame.mixer.init()

def decir_instruccion(texto):
    print(f"🔊 Generando audio: '{texto}'")
    
    try:
        # Generar el audio con gTTS en español ('es')
        tts = gTTS(text=texto, lang='es', slow=False)
        archivo_audio = "instruccion_temp.mp3"
        
        # Guardar el archivo mp3
        tts.save(archivo_audio)
        
        # Cargar y reproducir el archivo con pygame
        pygame.mixer.music.load(archivo_audio)
        pygame.mixer.music.play()
        
        # Esperar a que el audio termine de reproducirse antes de continuar
        while pygame.mixer.music.get_busy():
            time.sleep(0.1)
            
        # Opcional: descargar el archivo de la memoria para poder sobreescribirlo en el siguiente paso
        pygame.mixer.music.unload()
        
    except Exception as e:
        print(f"Error al generar o reproducir la voz: {e}")

# 2. Función simulada para obtener tu GPS
def obtener_mi_ubicacion_actual():
    # Coordenada simulada de ejemplo (Latitud, Longitud)
    return (39.4699, -0.37628) 

def iniciar_navegacion(pasos_de_la_ruta):
    print("Iniciando navegación...")
    
    for paso in pasos_de_la_ruta:
        instruccion = paso['indicacion']
        coordenadas_maniobra = (paso['end_location']['lat'], paso['end_location']['lng'])
        
        maniobra_completada = False
        
        while not maniobra_completada:
            mi_ubicacion = obtener_mi_ubicacion_actual()
            
            # Calcular distancia en metros usando geopy
            distancia_metros = geodesic(mi_ubicacion, coordenadas_maniobra).meters
            
            print(f"Distancia al próximo giro: {distancia_metros:.0f} metros")
            
            # Si estamos a menos de 50 metros del giro, damos la instrucción por voz
            if distancia_metros < 50:
                decir_instruccion(instruccion)
                maniobra_completada = True # Salimos del bucle para pasar a la siguiente instrucción
            else:
                # Esperar 2 segundos antes de volver a consultar el GPS
                time.sleep(2)
                
    decir_instruccion("Has llegado a tu destino. Navegación finalizada.")

# --- Simulación de uso ---
pasos_ejemplo = [
    {
        "indicacion": "En cincuenta metros, gira a la derecha hacia Calle de la Paz",
        "end_location": {"lat": 39.4700, "lng": -0.37600} 
    },
    {
        "indicacion": "Continúa recto por la Plaza de la Reina",
        "end_location": {"lat": 39.4735, "lng": -0.37550}
    }
]

if __name__ == "__main__":
    iniciar_navegacion(pasos_ejemplo)

Iniciando navegación...
Distancia al próximo giro: 27 metros
🔊 Generando audio: 'En cincuenta metros, gira a la derecha hacia Calle de la Paz'
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo giro: 405 metros
Distancia al próximo 

KeyboardInterrupt: 